In [44]:
# IMPORTS NEEDED FOR CODE TO RUN
import csv
import numpy as np
import pandas as pd
import regex as re

In [45]:
"""
    Function for stripping column names as well as data values of redundant white spaces.
"""
def strip_df_whitespaces(dataframe):
    # Strip column names of whitepaces
    column_names = [column.strip() for column in dataframe.columns]
    dataframe.columns = column_names

    # Strip datavalues of whitespaces
    dataframe = dataframe.map(lambda x: x.strip() if isinstance(x, str) else x)

    # Return result
    return dataframe

In [46]:
# Load in the uncleaned tabloid data
uncleaned_coverage_data = open("..\data\own_coverage_uncleaned.txt", encoding='UTF-8').read()

print(uncleaned_coverage_data)

| Category | # | Description | Implementation-Status | MAS-Status |
|----------|----------|----------|----------|----------|
| STORAGE |  |  |  |  |
|  | [0001](https://mas.owasp.org/MASWE/MASVS-STORAGE/MASWE-0001/) | Insertion of Sensitive Data into Logs |1| Beta |
|  | [0002](https://mas.owasp.org/MASWE/MASVS-STORAGE/MASWE-0002/) | Sensitive Data Stored With Insufficient Access Restrictions in Internal Locations |1| Placeholder |
|  | [0003](https://mas.owasp.org/MASWE/MASVS-STORAGE/MASWE-0003/) | Backup Unencrypted |1| Placeholder |
|  | [0004](https://mas.owasp.org/MASWE/MASVS-STORAGE/MASWE-0004/) | Sensitive Data Not Excluded From Backup |1| Beta |
|  | [0006](https://mas.owasp.org/MASWE/MASVS-STORAGE/MASWE-0006/) | Sensitive Data Stored Unencrypted in Private Storage Locations |1| Beta |
|  | [0007](https://mas.owasp.org/MASWE/MASVS-STORAGE/MASWE-0007/) | Sensitive Data Stored Unencrypted in Shared Storage Requiring No User Interaction |1| Beta |
| CRYPTO |  |  |  |  |
|  | [0009

<>:2: SyntaxWarning: invalid escape sequence '\d'
<>:2: SyntaxWarning: invalid escape sequence '\d'
C:\Users\Domi\AppData\Local\Temp\ipykernel_16984\304693285.py:2: SyntaxWarning: invalid escape sequence '\d'
  uncleaned_coverage_data = open("..\data\own_coverage_uncleaned.txt", encoding='UTF-8').read()


In [47]:
# Use regular expressions to reformat app names
# Regex pattern: replaces [AppName](https://github.com/anything) → AppName
uncleaned_coverage_data = re.sub(r"\[([^\]]+)\]\(https://mas.owasp\.org/[^\)]+\)", r"\1", uncleaned_coverage_data)
display(uncleaned_coverage_data)

'| Category | # | Description | Implementation-Status | MAS-Status |\n|----------|----------|----------|----------|----------|\n| STORAGE |  |  |  |  |\n|  | 0001 | Insertion of Sensitive Data into Logs |1| Beta |\n|  | 0002 | Sensitive Data Stored With Insufficient Access Restrictions in Internal Locations |1| Placeholder |\n|  | 0003 | Backup Unencrypted |1| Placeholder |\n|  | 0004 | Sensitive Data Not Excluded From Backup |1| Beta |\n|  | 0006 | Sensitive Data Stored Unencrypted in Private Storage Locations |1| Beta |\n|  | 0007 | Sensitive Data Stored Unencrypted in Shared Storage Requiring No User Interaction |1| Beta |\n| CRYPTO |  |  |  |  |\n|  | 0009 | Improper Cryptographic Key Generation |1| Beta |\n|  | 0010 | Improper Cryptographic Key Derivation |1| Placeholder |\n|  | 0011 | Cryptographic Key Rotation Not Implemented |1| Placeholder |\n|  | 0012 | Insecure or Wrong Usage of Cryptographic Key |1| Placeholder |\n|  | 0013 | Hardcoded Cryptographic Keys in Use |0| DEPRECAT

In [48]:
# Split data into seperate rows
specific_data_rows = uncleaned_coverage_data.split("\n")

# Store data in a csv file, using the "|" as seperators
with open("../data/own_coverage_data_uncleaned.csv", "w", newline="", encoding='UTF-8') as file:
    writer = csv.writer(file)
    for row in specific_data_rows:
        # Split rows on "|" and write them to the csv file
        writer.writerow(row.split("|"))

In [49]:
# Convert it to a pandas dataframe
coverage_data_df = pd.read_csv("../data/own_coverage_data_uncleaned.csv")

# Drop first and last columns
coverage_data_df.drop(["Unnamed: 0", "Unnamed: 6"], axis=1, inplace=True)
# Drop first row
coverage_data_df.drop(0, inplace=True)

# Strip column names of whitespaces
coverage_data_df = strip_df_whitespaces(coverage_data_df)

# Add uniform missing values sign
for column in coverage_data_df.columns:
    coverage_data_df.loc[coverage_data_df[column] == "", column] = "-"

# Reset the index to a meaningful one, considering the now removed values
coverage_data_df.reset_index(drop=True, inplace=True)

# Drop missing values
coverage_data_df.dropna(inplace=True)

display(coverage_data_df)

,Category,#,Description,Implementation-Status,MAS-Status
0,STORAGE,-,-,-,-
1,-,0001,Insertion of Sensitive Data into Logs,1,Beta
2,-,0002,Sensitive Data Stored With Insufficient Access...,1,Placeholder
3,-,0003,Backup Unencrypted,1,Placeholder
4,-,0004,Sensitive Data Not Excluded From Backup,1,Beta
...,...,...,...,...,...
120,-,0112,Inadequate Data Collection Declarations,0,Beta
121,-,0113,Lack of Proper Data Management Controls,0,Beta
122,-,0114,Inadequate Data Visibility Controls,0,Beta
123,-,0115,Inadequate or Ambiguous User Consent Mechanisms,0,Beta


In [50]:
# Add correct "App Name" value to every row
app_names = coverage_data_df.loc[coverage_data_df["Category"] != "-", "Category"].values
app_names_indexes = []
for app_name in app_names:
    app_names_indexes.append(coverage_data_df[coverage_data_df["Category"] == app_name].index[0])
# Add last index of dataframe rows as guardian value for next operation
app_names_indexes.append(len(coverage_data_df.values))


# Go through all app name indexes, locate the rows which require the same app name and fill them in
for index in range(len(app_names_indexes)-1):
    coverage_data_df.iloc[app_names_indexes[index]:app_names_indexes[index+1], 0] = app_names[index]

# Remove rows which only hold the category title and no actual vulnerability (8 of them in total)

# Drop non vulnerability rows
coverage_data_df.drop(coverage_data_df.loc[coverage_data_df["Description"] == "-"].index, inplace=True)

# Count amount of "-" missing values
print(np.sum(coverage_data_df["Description"] == "-"))

# Reset the index to a meaningful one, considering the now removed values
coverage_data_df.reset_index(drop=True, inplace=True)
display(coverage_data_df)

0


,Category,#,Description,Implementation-Status,MAS-Status
0,STORAGE,0001,Insertion of Sensitive Data into Logs,1,Beta
1,STORAGE,0002,Sensitive Data Stored With Insufficient Access...,1,Placeholder
2,STORAGE,0003,Backup Unencrypted,1,Placeholder
3,STORAGE,0004,Sensitive Data Not Excluded From Backup,1,Beta
4,STORAGE,0006,Sensitive Data Stored Unencrypted in Private S...,1,Beta
...,...,...,...,...,...
112,PRIVACY,0112,Inadequate Data Collection Declarations,0,Beta
113,PRIVACY,0113,Lack of Proper Data Management Controls,0,Beta
114,PRIVACY,0114,Inadequate Data Visibility Controls,0,Beta
115,PRIVACY,0115,Inadequate or Ambiguous User Consent Mechanisms,0,Beta


In [51]:
#Extract the only necessary column, as to compare it to the one in specific_data_cleaned.csv
coverage_data_df["Vulnerability"] = "MASWE-" + coverage_data_df["#"] + ": " + coverage_data_df["Description"]

# Drop all unimplemented vulnerabilities
coverage_data_df = coverage_data_df[coverage_data_df["Implementation-Status"] == "1"]

# Drop all other columns
#coverage_data_df.drop(coverage_data_df.columns[1:-1], axis=1, inplace=True)
coverage_data_df = coverage_data_df[["Category", "Vulnerability"]]

# Reset the index to a meaningful one, considering the now removed values
coverage_data_df.reset_index(drop=True, inplace=True)
display(coverage_data_df)

,Category,Vulnerability
0,STORAGE,MASWE-0001: Insertion of Sensitive Data into Logs
1,STORAGE,MASWE-0002: Sensitive Data Stored With Insuffi...
2,STORAGE,MASWE-0003: Backup Unencrypted
3,STORAGE,MASWE-0004: Sensitive Data Not Excluded From B...
4,STORAGE,MASWE-0006: Sensitive Data Stored Unencrypted ...
5,STORAGE,MASWE-0007: Sensitive Data Stored Unencrypted ...
6,CRYPTO,MASWE-0009: Improper Cryptographic Key Generation
7,CRYPTO,MASWE-0010: Improper Cryptographic Key Derivation
8,CRYPTO,MASWE-0011: Cryptographic Key Rotation Not Imp...
9,CRYPTO,MASWE-0012: Insecure or Wrong Usage of Cryptog...


In [52]:
# Export cleaned df to csv
coverage_data_df.to_csv("../data/own_coverage_data_cleaned.csv", index=False)